In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("data/sp500_rl_ready.parquet")
df.head()

,date,ticker,open_norm,high_norm,low_norm,close_norm,adj_close_norm,BB_Upper_norm,BB_Lower_norm,VWAP_norm,...,ATR_norm,NATR_norm,BB_Width_norm,volume_norm,returns,log_returns,excess_returns,market_return,volatility_5d,volatility_20d
0,1990-02-16,ABT,0.001321,0.001110,0.001073,0.000966,0.000435,NaN,NaN,0.002721,...,-6.104310,-2.134929,-2.058693,0.143253,NaN,NaN,NaN,NaN,NaN,NaN
1,1990-02-19,ABT,0.001321,0.001110,0.001073,0.000966,0.000435,0.000000,0.000689,0.002721,...,-3.994406,0.556056,-2.058693,-5.189859,0.000000,0.000000,0.000000,0.000000,NaN,NaN
2,1990-02-20,ABT,0.000965,0.000656,0.000716,0.000559,0.000252,0.000146,0.000399,0.001945,...,-3.549510,0.830383,-1.241963,0.109990,-0.015296,-0.015415,-0.000530,-0.014766,0.010816,0.010816
3,1990-02-21,ABT,0.000356,0.000252,0.000307,0.000305,0.000137,0.000168,0.000249,0.001195,...,-3.358579,0.943357,-0.929560,0.099023,-0.009709,-0.009756,-0.003908,-0.005801,0.007740,0.007740
4,1990-02-22,ABT,0.000508,0.000555,0.000562,0.000407,0.000183,0.000129,0.000235,0.001032,...,-3.236351,1.003436,-0.977109,0.133421,0.003922,0.003914,0.008865,-0.004944,0.008803,0.008803


In [3]:
nan_counts = df.isna().sum()
print(nan_counts)

date                     0
ticker                   0
open_norm                0
high_norm                0
low_norm                 0
close_norm               0
adj_close_norm           0
BB_Upper_norm           30
BB_Lower_norm           30
VWAP_norm                0
RSI_norm                84
KDJ_K_norm               0
KDJ_D_norm               0
MFI_norm                 0
STOCHF_K_norm            0
STOCHF_D_norm            0
BB_Position_norm         0
MACD_norm                0
MACD_Signal_norm         0
MACD_Histogram_norm      0
KDJ_J_norm               0
WILLIAMS_R_norm          0
AROONOSC_norm            0
CCI_norm                60
CMO_norm                 0
ROC_norm               300
ATR_norm                 0
NATR_norm                0
BB_Width_norm            0
volume_norm              0
returns                 30
log_returns             30
excess_returns          30
market_return           30
volatility_5d           60
volatility_20d          60
dtype: int64


In [4]:
max(nan_counts)

300

In [ ]:
"""
Debug version to understand why short positions are being zeroed out.
"""

import numpy as np

def debug_validate_action(action, n_assets=30, max_short_ratio=0.3):
    """Debug version of action validation with detailed logging."""
    
    print(f"\n=== DEBUG ACTION VALIDATION ===")
    action = np.array(action, dtype=np.float32)
    
    # Split into long and short components
    long_weights = action[:n_assets].copy()
    short_weights = action[n_assets:].copy()
    
    print(f"Original action:")
    print(f"  Long (first 5): {long_weights[:5]}")
    print(f"  Short (first 5): {short_weights[:5]}")
    print(f"  Long sum: {np.sum(long_weights)}")
    print(f"  Short sum: {np.sum(short_weights)}")
    
    # Check for conflicts
    conflicts = []
    for i in range(n_assets):
        if long_weights[i] > 1e-8 and short_weights[i] < -1e-8:
            conflicts.append((i, long_weights[i], short_weights[i]))
    
    print(f"Conflicts found: {len(conflicts)}")
    for i, long_val, short_val in conflicts:
        print(f"  Asset {i}: long={long_val:.6f}, short={short_val:.6f}")
    
    # Constraint 1: Resolve conflicts
    for i in range(n_assets):
        if long_weights[i] > 1e-8 and short_weights[i] < -1e-8:
            if abs(long_weights[i]) >= abs(short_weights[i]):
                print(f"  Resolving conflict {i}: keeping long={long_weights[i]:.6f}, removing short={short_weights[i]:.6f}")
                short_weights[i] = 0.0
            else:
                print(f"  Resolving conflict {i}: keeping short={short_weights[i]:.6f}, removing long={long_weights[i]:.6f}")
                long_weights[i] = 0.0
    
    print(f"After conflict resolution:")
    print(f"  Long sum: {np.sum(long_weights)}")
    print(f"  Short sum: {np.sum(short_weights)}")
    
    # Constraint 2: Long weights sum to 1
    long_weights = np.maximum(long_weights, 0.0)
    long_sum = np.sum(long_weights)
    if long_sum > 1e-8:
        long_weights = long_weights / long_sum
        print(f"Long weights normalized to sum=1")
    else:
        long_weights = np.ones(n_assets) / n_assets
        print(f"No long weights, using equal allocation")
    
    # Constraint 3: Short weights respect max ratio
    short_weights = np.minimum(short_weights, 0.0)
    short_sum = np.sum(short_weights)
    print(f"Short sum before scaling: {short_sum}")
    
    if short_sum < -1e-8:
        current_short_ratio = abs(short_sum)
        print(f"Current short ratio: {current_short_ratio}")
        if current_short_ratio > max_short_ratio:
            scale_factor = max_short_ratio / current_short_ratio
            short_weights = short_weights * scale_factor
            print(f"Scaled shorts by factor {scale_factor}")
        else:
            print(f"Short ratio within limits")
    else:
        print(f"No short positions")
    
    print(f"Final result:")
    print(f"  Long sum: {np.sum(long_weights)}")
    print(f"  Short sum: {np.sum(short_weights)}")
    print(f"  Net sum: {np.sum(long_weights) + np.sum(short_weights)}")
    
    return np.concatenate([long_weights, short_weights])

# Test realistic short-selling scenarios (no conflicts)
print("\n" + "="*60)
print("REALISTIC SHORT-SELLING TEST CASES")
print("="*60)

print("\n=== REALISTIC TEST 1: No Conflicts ===")
# Long some assets, short different assets
realistic_action_1 = np.concatenate([
    [0.6, 0.4, 0.0, 0.0],  # Long on assets 0,1 only
    [0.0, 0.0, -0.2, -0.1] # Short on assets 2,3 only (no conflicts!)
])
result_real_1 = debug_validate_action(realistic_action_1, 4, 0.3)

print("\n=== REALISTIC TEST 2: Mixed Portfolio ===")
# More complex but realistic allocation
realistic_action_2 = np.concatenate([
    [0.3, 0.3, 0.2, 0.2],  # Long positions on all assets
    [0.0, 0.0, 0.0, 0.0]   # No short positions
])
result_real_2 = debug_validate_action(realistic_action_2, 4, 0.3)

print("\n=== REALISTIC TEST 3: Aggressive Short Strategy ===")
# Long some, short others, at the limit
realistic_action_3 = np.concatenate([
    [0.5, 0.5, 0.0, 0.0],  # Long 50-50 on first two
    [0.0, 0.0, -0.15, -0.15] # Short 15% each on last two (total 30%)
])
result_real_3 = debug_validate_action(realistic_action_3, 4, 0.3)

print("\n=== REALISTIC TEST 4: Excessive Shorts (No Conflicts) ===")
# Test the scaling when short ratio exceeds limit
realistic_action_4 = np.concatenate([
    [1.0, 0.0, 0.0, 0.0],  # All long in first asset
    [0.0, -0.2, -0.2, -0.2] # Shorts total 60% > 30% limit
])
result_real_4 = debug_validate_action(realistic_action_4, 4, 0.3)

print("\n" + "="*60)
print("SUMMARY:")
print("- Realistic Test 1: Should preserve all positions (no conflicts)")
print("- Realistic Test 2: Long-only case")  
print("- Realistic Test 3: Balanced long/short within limits")
print("- Realistic Test 4: Short scaling from 60% down to 30%")
print("="*60)

=== TEST CASE 1: Normal Action ===

=== DEBUG ACTION VALIDATION ===
Original action:
  Long (first 5): [0.4 0.3 0.2 0.1]
  Short (first 5): [ 0.   0.  -0.2  0. ]
  Long sum: 1.0
  Short sum: -0.20000000298023224
Conflicts found: 1
  Asset 2: long=0.200000, short=-0.200000
  Resolving conflict 2: keeping long=0.200000, removing short=-0.200000
After conflict resolution:
  Long sum: 1.0
  Short sum: 0.0
Long weights normalized to sum=1
Short sum before scaling: 0.0
No short positions
Final result:
  Long sum: 1.0
  Short sum: 0.0
  Net sum: 1.0

=== TEST CASE 2: Conflicting Positions ===

=== DEBUG ACTION VALIDATION ===
Original action:
  Long (first 5): [0.5 0.3 0.2 0. ]
  Short (first 5): [ 0.  -0.1 -0.2  0. ]
  Long sum: 1.0
  Short sum: -0.30000001192092896
Conflicts found: 2
  Asset 1: long=0.300000, short=-0.100000
  Asset 2: long=0.200000, short=-0.200000
  Resolving conflict 1: keeping long=0.300000, removing short=-0.100000
  Resolving conflict 2: keeping long=0.200000, removing